<a href="https://colab.research.google.com/github/csiyu0962-collab/DataScience-GenAI-Submissions/blob/main/7_02_Machine_Learning_Modelling_Student_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [2]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [4]:
x_values = df.drop(['loan_status'], axis = 1)
print(f'X values: \n {x_values.head()}\n')
print('\n')

y_value = df['loan_status']
print(f'Y value: \n {y_value[0:5]}')

X values: 
    age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amount  interest_rate  debt_to

## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [5]:
# split data into training and test
from sklearn.model_selection  import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(x_values, y_value, test_size = 0.2, random_state=1234)

# print the shapes to check everything is OK
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(40000, 15)
(10000, 15)
(40000,)
(10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [8]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. 创建 MinMaxScaler 对象
scaler = MinMaxScaler()

# 2. 仅在训练集上进行拟合和转换
X_train_scaled = scaler.fit_transform(X_train)
# 使用 X_train 的原始列名重建 DataFrame
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

# 3. 仅对测试集进行转换（不重新 fit）
X_test_scaled = scaler.transform(X_test)
# 使用 X_test 的原始列名重建 DataFrame
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# 查看缩放后的训练集前几行
X_train_scaled_df.head()

,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio
37174,0.230769,0.050125,0.236723,0.547809,0.080000,0.025303,0.036342,0.0,0.000000,0.0,0.335678,0.328033,0.104010,0.236829,0.236446
29950,0.461538,0.278195,0.130817,0.695219,0.453333,0.021167,0.084270,0.0,0.000000,0.0,0.128643,0.393404,0.375940,0.141997,0.141566
49833,0.173077,0.022556,0.010860,0.617530,0.100000,0.000253,0.045871,0.0,0.111111,0.0,0.144724,0.240872,0.536341,0.421977,0.421687
43672,0.461538,0.122807,0.314430,0.645418,0.203333,0.003427,0.188077,0.0,0.000000,0.0,0.200000,0.199647,0.431078,0.110888,0.109940
23669,0.019231,0.025063,0.489830,0.501992,0.013333,0.000070,0.102319,0.0,0.111111,0.0,1.000000,0.499411,0.159148,0.381836,0.381024


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [10]:
from sklearn.linear_model import LogisticRegression as LogR
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.ensemble import GradientBoostingClassifier as GBDT
from xgboost import XGBClassifier as XGB
from sklearn.metrics import precision_recall_fscore_support

LogR_algo = LogR()
LogR_model = LogR_algo.fit(X_train, Y_train)

RF_algo = RF()
RF_model = RF_algo.fit(X_train, Y_train)

GBDT_algo = GBDT()
GBDT_model = GBDT_algo.fit(X_train, Y_train)

XGB_algo = XGB()
XGB_model = XGB_algo.fit(X_train, Y_train)

models = [LogR_model, RF_model, GBDT_model, XGB_model]
names = ['Logistic Regression', 'Random Forest', 'GBDT', 'XGBDT']


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [11]:
for i in range(4):
  print(f"Model: {names[i]}")

  # predict based on training data
  predict = models[i].predict(X_train)

  # Calculate precision, recall, and F1-score
  precision, recall, f1_score, _ = precision_recall_fscore_support(Y_train, predict, average='macro')
  print(f"Macro Precision: {precision}")
  print(f"Macro Recall: {recall}")
  print(f"Macro F1-score: {f1_score}")
  print("\n")

Model: Logistic Regression
Macro Precision: 0.7236819509089856
Macro Recall: 0.717219454785451
Macro F1-score: 0.7186902718282036


Model: Random Forest
Macro Precision: 1.0
Macro Recall: 1.0
Macro F1-score: 1.0


Model: GBDT
Macro Precision: 0.8928352200291871
Macro Recall: 0.8892832841475249
Macro F1-score: 0.8907308891702573


Model: XGBDT
Macro Precision: 0.9379619754676296
Macro Recall: 0.9359079130493941
Macro F1-score: 0.9368279553325759




## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?